## **LangChain 입문** 

1. LangChain의 역할과 LLM의 SDK를 사용하는 경우의 차이를 설명한다.
2. `ChatOpenAI`와 `ChatOllama`로 모델을 호출한다.
3. `SystemMessage` / `HumanMessage` / `AIMessage`의 역할을 구분한다.
4. `PromptTemplate`과 `ChatPromptTemplate`을 작성한다.
5. `prompt | model | parser` 체인의 입력·출력 타입을 설명하고 실행한다.

> **한눈에 보기**
> - LangChain = 모델이 아니라 **LLM 앱 부품을 연결하는 프레임워크**. OpenAI SDK 대신 공통 `invoke()`로 호출
> - `ChatOpenAI`(gpt-4o-mini)와 `ChatOllama`(로컬 EXAONE 3.5)를 같은 코드로 호출, 결과는 `AIMessage`
> - `SystemMessage`/`HumanMessage`로 역할 나누기, `PromptTemplate`/`ChatPromptTemplate`로 프롬프트 재사용
> - `prompt | llm | StrOutputParser()` 체인 — 모델만 바꿔 끼워 재사용
> - 정리 문서: [LangChain입문.md](LangChain입문.md) · 슬라이드 이미지는 자리만 표시, 📝 정리 셀은 실행 결과를 보고 직접 정리

## 0. LangChain이란?

**LangChain은 LLM을 이용한 애플리케이션을 만들 때 필요한 구성요소를 연결하는 Python/JavaScript 프레임워크**입니다. 모델 자체를 학습시키는 도구가 아니라, 이미 제공되는 모델을 프로그램 안에서 **호출하고, 입력을 구성하고, 출력을 처리하고, 다른 기능과 결합**하는 데 도움을 줍니다.

### 사용하는 이유

| 장점 | 무엇이 편리해지나? | 이 노트북에서 확인할 API |
|---|---|---|
| **모델 인터페이스의 일관성** | OpenAI와 로컬 Ollama 모델을 비슷한 코드로 호출 | `ChatOpenAI`, `ChatOllama`, `invoke()` |
| **프롬프트 재사용** | 입력 변수만 바꾸어 같은 질문 구조를 재사용 | `PromptTemplate`, `ChatPromptTemplate` |
| **역할 기반 대화** | system/human/AI 메시지를 명확히 구분 | `SystemMessage`, `HumanMessage`, `AIMessage` |
| **모듈형 조합** | 프롬프트 → 모델 → 파서를 손쉽게 연결 | `|` 연산자(LCEL), `StrOutputParser` |
| **기능 확장성** | 이후 검색(RAG), 도구 호출, Agent로 확장 가능 | 이번 노트북에서는 **기초만** 다룸 |

### LangChain이 반드시 필요한가?

**아닙니다.** 단순히 질문 1개를 LLM에 전달한다면 **OpenAI SDK만으로도 충분**합니다. 여러 모델을 교체하거나 프롬프트·체인·검색 등을 구조화해 재사용할 때 LangChain의 이점이 커집니다. 프레임워크를 쓰면 의존성이 늘어나고, 복잡한 응용에서는 디버깅할 계층도 늘어날 수 있습니다.

### 전체 구조

```text
Python 애플리케이션
 └─ LangChain (프롬프트, 메시지, Runnable/LCEL, 파서)
     ├─ langchain-openai  → OpenAI API → gpt-4o-mini (원격)
     └─ langchain-ollama  → Ollama 서버 → exaone3.5 (로컬)
```

- `langchain`: 상위 프레임워크 기능
- `langchain-core`: 프롬프트, 메시지, Runnable 등 핵심 추상화
- `langchain-openai`: OpenAI 연동
- `langchain-ollama`: Ollama 연동

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

### 0-1. OpenAI SDK와 LangChain 비교

**OpenAI SDK 직접 호출:**

```python
from openai import OpenAI
client = OpenAI()
response = client.responses.create(
    model="gpt-4o-mini",
    input="LangChain이란 무엇인가요?"
)
print(response.output_text)
```

**LangChain 호출:**

```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")
aim = llm.invoke("LangChain이란 무엇인가요?")
print(aim.content)
```

LangChain 채팅 모델의 `invoke()`는 일반적으로 `AIMessage`를 반환합니다. 

###  필수 용어

| 용어 | 뜻 |
|---|---|
| **LLM** | 텍스트 등을 생성하는 언어 모델 |
| **Chat model** | 역할이 있는 메시지를 입력받고 메시지를 응답하는 모델 인터페이스 |
| **Prompt** | 모델에 주는 지시·질문·맥락 |
| **Message** | 역할(role)과 내용(content)을 포함한 대화 단위 |
| **Runnable** | `invoke()`, `batch()`, `stream()` 등 공통 호출 방식을 갖는 실행 구성요소 |
| **LCEL** | Runnable을 파이프(\|)로 연결하는 표현 방식 |
| **Output parser** | 모델 출력을 프로그램이 사용할 형태로 변환하는 구성요소 |

### 실행 방식 기억하기

```text
invoke(x)  : 단일 입력 → 단일 결과
batch(xs)  : 여러 입력 → 여러 결과
stream(x)  : 결과를 조각(토큰) 단위로 순회(모델/구성요소의 지원에 따라 다름)
```

> `invoke()` 한 번 호출했다고 모델이 이전 질문을 자동으로 기억하지는 않습니다. 이전 내용을 기억시키려면 메시지 기록을 직접 전달하거나 이후 수업에서 상태 관리 기능을 사용해야 합니다.

In [2]:
%pip install -U langchain langchain-core langchain-openai langchain-ollama langchain-mcp-adapters

Note: you may need to restart the kernel to use updated packages.


In [3]:
# 설치된 패키지 버전 확인 (환경에 따라 버전이 다를 수 있음)
from importlib.metadata import version
for package in ["langchain", "langchain-core", "langchain-openai", "langchain-ollama"]:
    print(f"{package}: {version(package)}")

langchain: 1.4.6
langchain-core: 1.6.12
langchain-openai: 1.7.2
langchain-ollama: 1.1.0


In [4]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama

llm_openai = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
llm_exaone = ChatOllama(model="exaone3.5", temperature=0.7)
print("모델 객체 준비 완료 (이 단계에서는 API를 호출하지 않습니다)")

모델 객체 준비 완료 (이 단계에서는 API를 호출하지 않습니다)


**📝 정리** — 모델 객체를 만드는 것만으로는 API를 호출하지 않는다. 실제 호출은 `invoke()` 때 일어난다.

### 1. 기본 LLM 호출 (`invoke`)

- LangChain의 채팅 모델은 공통 `invoke()` 인터페이스로 호출할 수 있습니다. 
- 문자열을 그대로 넘기면 내부적으로 `HumanMessage`로 감싸서 모델에 전달되고, 결과는 `AIMessage` 객체로 돌아옵니다. 
- 실제 답변 텍스트는 `.content`에 담겨 있습니다.

**핵심**: OpenAI든 Ollama든 `invoke()` 하나로 동일하게 호출합니다 \
이것이LangChain이 제공하는 "통일된 인터페이스"의 가장 기본적인 예시입니다.


In [5]:
# gpt-4o-mini 버전
response = llm_openai.invoke("LangChain이 뭔지 한 문장으로 설명해줘")
print("[gpt-4o-mini]")
print(response.content)

[gpt-4o-mini]
LangChain은 다양한 언어 모델과 애플리케이션을 통합하여 자연어 처리(NLP) 작업을 쉽게 구축하고 확장할 수 있도록 돕는 프레임워크입니다.


In [6]:
# EXAONE 3.5 버전 (Ollama)
response = llm_exaone.invoke("LangChain이 뭔지 한 문장으로 설명해줘")
print("[exaone3.5]")
print(response.content)

[exaone3.5]
LangChain은 다양한 언어 모델을 통합하고 활용하여 복잡한 텍스트 작업을 단순화하는 오픈 소스 프레임워크입니다.


**📝 정리** — 같은 질문을 GPT와 EXAONE에 `invoke()` 하나로 보냈다. 회사가 다른 모델도 호출 코드가 같다는 것이 LangChain의 첫 번째 장점이다.

##### OpenAI 클래스 (텍스트 완성형)
- 과거 문장 뒤에 이어질 말(Completion)을 예측하던 시절의 구형 모델들을 다루는 클래스입니다. 
- 프롬프트 하나를 던지면 뒷문장을 이어 써주는 정직한 구조입니다.
- 대화의 맥락(System, User, Assistant 구분)을 인지하지 못하며, `gpt-4o-mini`는 이 레거시 Completion 인터페이스가 아니라 채팅 모델로 호출해야 합니다.

##### **ChatOpenAI 클래스 (대화형 - ★추천)**
- 현재 우리가 쓰는 ChatGPT처럼 대화(Chat)를 주고받도록 튜닝된 최신 모델들을 다루는 클래스입니다. 
- 시스템 지침(System), 유저 질문(User), AI 답변(Assistant)으로 역할을 명확히 나누어 입력할 수 있습니다.
- Tool Calling 및 Structured Output은 지원 모델과 적합한 통합 인터페이스를 통해 구현할 수 있습니다. 

In [7]:
# gpt-4o-mini는 구형 Completion용 OpenAI 클래스 대신 ChatOpenAI를 사용합니다.
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")
response = llm.invoke("한국의 수도는 어디인가요?")
print(type(response))  # AIMessage
print(response.content)  # 답변 텍스트

<class 'langchain_core.messages.ai.AIMessage'>
한국의 수도는 서울입니다.


In [8]:
# 문자열의 뒤를 자동으로 이어 쓰는 Completion과 달리, Chat 모델은 질문을 대화 메시지로 해석합니다.
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")
response = llm.invoke("'한국은 '으로 시작하는 문장을 한 문장 만들어줘.")
print(response.content)

한국은 아름다운 자연경관과 풍부한 문화유산이 어우러진 나라입니다.


In [9]:
from langchain_openai import ChatOpenAI

# LLM 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

# 모델에 텍스트 생성 요청하기
response = llm.invoke("한국의 수도는 어디인가요?")

# 응답 객체 출력 
print(type(response))
print(response)

<class 'langchain_core.messages.ai.AIMessage'>
content='한국의 수도는 서울입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 15, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4be5992575', 'id': 'chatcmpl-EaX8K5wiUg3nPg6dlY6EqgClhpeFk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a151ed-79bc-7ab0-8cd5-372967442c9c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 7, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0

**📝 정리** — `AIMessage`를 통째로 출력하면 답(`content`)뿐 아니라 `response_metadata`에 토큰 사용량(`token_usage`), 모델 이름 등이 들어 있다. 비용 계산할 때 여기를 본다.

In [10]:
from langchain_openai import ChatOpenAI

# 채팅 모델 초기화
llm = ChatOpenAI(model='gpt-4o-mini')

# 모델에 직접 텍스트를 입력
result = llm.invoke("김치찌개에 대한 레시피를 3단계로 설명해주세요.")

# 답변을 출력 
print(result.content)

김치찌개를 만드는 방법을 3단계로 간단히 설명해 드릴게요.

### 1단계: 재료 준비
- **필요한 재료**: 
  - 익은 김치 (2컵)
  - 돼지고기 (목살이나 삼겹살, 약 200g)
  - 두부 (1/2모)
  - 대파 (1대)
  - 마늘 (다진 것 1큰술)
  - 고춧가루 (1-2큰술, 기호에 따라 조절)
  - 소금, 후추
  - 물 (3컵 정도)

### 2단계: 재료 볶기
- **조리 방법**: 
  1. 적당한 크기로 썬 돼지고기를 냄비에 넣고 중불로 볶아줍니다. 고기가 반쯤 익으면 다진 마늘과 고춧가루를 넣고 볶아 향을 낸 후, 익은 김치를 추가하고 함께 볶습니다.
  2. 김치가 살짝 익으면 물을 붓고 끓입니다.

### 3단계: 끓이기 및 마무리
- **완성하기**: 
  1. 끓기 시작하면 불을 줄이고 약 15-20분 정도 더 끓여서 국물의 맛이 우러나게 합니다. 
  2. 그 후, 두부를 적당한 크기로 자르고 넣어줍니다. 대파와 후추를 추가하고, 간을 보면서 소금으로 간을 맞춥니다. 
  3. 마지막으로 5분 정도 더 끓인 후 불을 끄고, 그릇에 담아 맛있게 즐기세요!

이렇게 하면 기본적인 김치찌개가 완성됩니다. 취향에 따라 추가 재료를 넣어 보시거나 맛을 조절해보세요!


---

#### 2. 메시지 기반 호출 (`SystemMessage` / `HumanMessage`)

실제 서비스에서는 문자열 하나만 던지기보다, **역할이 구분된 메시지 리스트**를
전달하는 경우가 대부분이다.

- `SystemMessage`: 모델의 역할/말투/제약을 지시 (사용자에게는 보이지 않는 지시문)
- `HumanMessage`: 실제 사용자의 질문
- `AIMessage`: 모델의 이전 답변 — 대화 기록을 구성할 때 사용

시스템 메시지를 바꾸면 같은 질문에도 답변 스타일이 크게 달라지는 것을 확인해보자.


In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# 최신 대화형 모델 사용 (기본값)
chat = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="너는 초등학생의 눈높이에 맞춰 답변하는 친절한 선생님이야."),
    HumanMessage(content="하늘은 왜 파란색이에요?")
]

result = chat.invoke(messages)
print(type(result))  # AIMessage 타입
print(result)        # 메타데이터 포함 객체
print(result.content)

<class 'langchain_core.messages.ai.AIMessage'>
content='하늘이 파란색인 이유는 빛의 성질과 관련이 있어요. 태양빛은 여러 가지 색깔의 빛으로 이루어져 있는데, 이 색깔들은 서로 다른 파장을 가지고 있어요. \n\n하늘에 있는 공기 중의 아주 작은 입자들이 태양빛을 흩어지게 만드는데, 이때 짧은 파장의 파란색 빛이 가장 많이 흩어져요. 그래서 우리가 하늘을 볼 때, 파란색이 더 많이 보이게 되는 거죠. \n\n이렇게 해서 맑은 날에는 하늘이 파랗게 보이는 거예요! 너무 멋지죠? ✨' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 148, 'prompt_tokens': 44, 'total_tokens': 192, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b7e99f0845', 'id': 'chatcmpl-EaX8PvtP3HACLbRxSzJxzbCn3zjbt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a151ed-8ef3-7f70-aa89-a2e6f

In [12]:
# gpt-4o-mini 버전
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="너는 20년차 파이썬 강사야. 답변은 항상 한국어로, 예시 코드를 곁들여 설명해."),
    HumanMessage(content="리스트 컴프리헨션이 뭐야?"),
]

response = llm_openai.invoke(messages)
print("[gpt-4o-mini]")
print(response.content)

[gpt-4o-mini]
리스트 컴프리헨션(List Comprehension)은 파이썬에서 리스트를 간결하게 생성하는 방법입니다. 일반적으로 리스트를 만들 때는 반복문을 사용해야 하지만, 리스트 컴프리헨션을 사용하면 더 간단하고 직관적으로 리스트를 생성할 수 있습니다.

기본적인 문법은 다음과 같습니다:

```python
[표현식 for 항목 in iterable if 조건]
```

- `표현식`: 리스트의 각 요소로 변환할 값
- `항목`: iterable에서 가져오는 각 요소
- `iterable`: 반복 가능한 객체(예: 리스트, 튜플, 문자열 등)
- `조건`: (선택적) 특정 조건을 만족하는 경우에만 포함

### 예시 1: 간단한 리스트 생성

1부터 10까지의 제곱수를 포함하는 리스트를 생성하고 싶다면, 다음과 같이 사용할 수 있습니다.

```python
squares = [x**2 for x in range(1, 11)]
print(squares)
```

출력:
```
[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
```

### 예시 2: 조건을 포함한 리스트 생성

짝수의 제곱만 포함하는 리스트를 생성하고 싶다면, 조건을 추가할 수 있습니다.

```python
even_squares = [x**2 for x in range(1, 11) if x % 2 == 0]
print(even_squares)
```

출력:
```
[4, 16, 36, 64, 100]
```

### 요약

리스트 컴프리헨션은 코드를 간결하게 만들고 가독성을 향상시키는 데 유용한 도구입니다. 위의 예시처럼 반복문과 조건을 사용하여 리스트를 쉽게 생성할 수 있으며, 이를 통해 코드의 길이를 줄이고 더 명확하게 표현할 수 있습니다.


In [13]:
# EXAONE 3.5 버전 (Ollama)
response = llm_exaone.invoke(messages)  # 위 셀에서 만든 messages를 그대로 재사용
print("[exaone3.5]")
print(response.content)

[exaone3.5]
리스트 컴프리헨션은 파이썬에서 매우 간결하고 효율적인 방법으로 리스트를 생성하는 문법입니다. 전통적인 `for` 루프를 사용하여 리스트를 만드는 것보다 더 짧고 읽기 쉬운 코드를 작성할 수 있게 해줍니다. 주로 다음과 같은 구조를 따릅니다:

```python
[표현식 for 항목 in 반복가능한 객체 if 조건]
```

### 기본 구조와 예시

1. **단순 리스트 생성**:
   만약 숫자 리스트를 생성하고 싶다면, 다음과 같이 할 수 있습니다.

   ```python
   # 1부터 5까지의 숫자 리스트 생성
   numbers = [x for x in range(1, 6)]
   print(numbers)  # 출력: [1, 2, 3, 4, 5]
   ```

2. **조건 추가**:
   특정 조건을 만족하는 항목들만 리스트에 포함시키고 싶다면 조건문을 추가할 수 있습니다.

   ```python
   # 짝수만 포함하는 리스트 생성
   even_numbers = [x for x in range(1, 11) if x % 2 == 0]
   print(even_numbers)  # 출력: [2, 4, 6, 8, 10]
   ```

3. **중첩 리스트 컴프리헨션**:
   더 복잡한 구조에서는 중첩된 리스트 컴프리헨션을 사용할 수도 있습니다. 예를 들어, 행렬(행렬의 리스트)을 생성할 때 유용합니다.

   ```python
   # 2x3 행렬 생성
   matrix = [[j * 3 + i + 1 for j in range(2)] for i in range(3)]
   print(matrix)
   # 출력:
   # [[1, 2], [4, 5]]
   ```

### 장점
- **간결성**: 긴 `for` 루프와 `append()` 메서드를 사용하는 것보다 훨씬 간결합니다.
- **성능**: 일반적으로 더 빠르게 실행됩니다. 특히 큰 데이터셋을 다룰 때 이점이 큽니다.
- **읽기 쉬움**: 코드의 의도를 더 명확하

**📝 정리** — 같은 `messages` 리스트를 두 모델에 그대로 넣었다. 둘 다 SystemMessage 지시대로 한국어 + 예시 코드로 답했다.

### 자료형으로 이해하는 PromptTemplate과 ChatPromptTemplate

```text
{"word": "재귀함수"}  (dict)
         ↓ PromptTemplate.invoke(...)
StringPromptValue (문자열 기반 프롬프트 객체)
         ↓ ChatOpenAI.invoke(...)
AIMessage (모델 답변 객체)
         ↓ .content 또는 StrOutputParser()
텍스트 콘텐츠 / str
```

```text
{"role": "DB", "question": "인덱스란?"}  (dict)
         ↓ ChatPromptTemplate.invoke(...)
ChatPromptValue (역할이 구분된 메시지 모음)
         ↓ ChatOpenAI.invoke(...)
AIMessage
```

`PromptValue`는 **프롬프트 내용을 감싼 객체**입니다. `prompt.to_string()`이나 `prompt.to_messages()`로 표현을 확인할 수 있습니다. `AIMessage.content`는 일반적으로 문자열이지만 모델/출력 형식에 따라 콘텐츠 블록 목록일 수도 있습니다.

#### 3. `PromptTemplate`으로 프롬프트 템플릿화

질문 문장을 매번 파이썬 문자열 포매팅(f-string)으로 직접 만드는 대신,
`PromptTemplate`을 쓰면 `{변수}` 자리표시자를 채워 넣는 방식으로
프롬프트를 재사용 가능한 "템플릿"으로 만들 수 있다.

강의 자료나 서비스 코드에서 **여러 개의 유사한 질문을 자동으로 생성**할 때
특히 유용하다.


In [14]:
# gpt-4o-mini 버전
from langchain_core.prompts import PromptTemplate

word_template = PromptTemplate.from_template(
    "다음 단어의 뜻을 초등학생도 이해할 수 있게 2~3문장으로 설명해줘: {word}"
)

prompt = word_template.invoke({"word": "재귀함수"})
response = llm_openai.invoke(prompt)
print("[gpt-4o-mini]")
print(response.content)

[gpt-4o-mini]
재귀함수는 어떤 함수가 자기 자신을 다시 부르는 기능을 가진 함수예요. 마치 거울 속의 거울처럼, 계속 같은 모습이 반복되는 것과 비슷해요. 예를 들어, 숫자를 세는 함수를 만들 때, 그 함수가 자기 자신을 호출해서 숫자를 하나씩 줄여가면서 세는 방식이에요.


In [15]:
# EXAONE 3.5 버전 (Ollama)
prompt = word_template.invoke({"word": "재귀함수"})  # 템플릿은 모델과 무관하게 재사용
response = llm_exaone.invoke(prompt)
print("[exaone3.5]")
print(response.content)

[exaone3.5]
재귀함수는 마치 자기 자신을 다시 부르는 마법 같은 컴퓨터 함수야. 예를 들어, "나를 세 번 말해봐"라고 부탁하면, 함수가 "나를"이라고 말하고, 그 다음에 다시 "나를"이라고 말하고, 마지막으로 또 "나를"이라고 말하는 것처럼, 자기 자신을 계속 호출해서 문제를 해결해 나가는 거야. 이렇게 하면 복잡한 일도 작은 단계로 나눠서 쉽게 처리할 수 있어!


**📝 정리** — 템플릿은 모델과 무관하다. 같은 `word_template`을 두 모델에 넣었고, "초등학생도 이해할 수 있게"라는 지시에 EXAONE이 더 아이 말투로 답했다.

In [16]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "{name}님, {topic}에 대해 {language}로 설명해주세요."
)

result = prompt.invoke({
    "name": "홍길동",
    "topic": "LangChain",
    "language": "한국어"
})

print(result.to_string())

홍길동님, LangChain에 대해 한국어로 설명해주세요.


### 4. `ChatPromptTemplate` (system + human 템플릿)

`PromptTemplate`이 문자열 하나를 다루는 템플릿이라면, `ChatPromptTemplate`은
system/human 등 **여러 역할의 메시지를 한 번에 템플릿화**할 수 있다.
`{role}`, `{question}`처럼 여러 자리표시자를 동시에 채울 수 있어서
실무 프롬프트 관리에 훨씬 자주 쓰인다.


In [17]:
# gpt-4o-mini 버전
from langchain_core.prompts import ChatPromptTemplate

qa_template = ChatPromptTemplate.from_messages([
    ("system", "너는 {role} 분야 전문가야. 답변은 한국어로, 3줄 이내로 핵심만 말해줘."),
    ("human", "{question}"),
])

prompt = qa_template.invoke({"role": "데이터베이스", "question": "인덱스는 왜 필요해?"})
response = llm_openai.invoke(prompt)
print("[gpt-4o-mini]")
print(response.content)

[gpt-4o-mini]
인덱스는 데이터베이스에서 검색 속도를 크게 향상시켜줍니다. 대량의 데이터에서 특정 값을 빠르게 찾을 수 있도록 도와주며, 효율적인 쿼리 성능을 제공합니다. 또한, 인덱스를 사용하면 정렬과 집계 작업도 더 빠르게 수행할 수 있습니다.


In [18]:
# EXAONE 3.5 버전 (Ollama)
prompt = qa_template.invoke({"role": "네트워크", "question": "TCP와 UDP의 차이는?"})
response = llm_exaone.invoke(prompt)
print("[exaone3.5]")
print(response.content)

[exaone3.5]
TCP는 신뢰성 있는 연결 지향적 통신을 제공하고, UDP는 빠른 전송을 위해 연결 없이 데이터를 보냅니다. 오류 검사와 순서 보장은 TCP가 우위입니다.


In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 채팅 모델 초기화
llm = ChatOpenAI(model='gpt-4o-mini')

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template("{food}에 대한 레시피를 3단계로 설명해주세요.")

# prompt + llm 조합으로 체인을 구성 
chain = prompt | llm

# 체인 실행
result = chain.invoke({"food": "비빔밥"})

# 결과 출력
print("답변: ", result.content)

답변:  비빔밥은 다양한 재료를 섞어 먹는 한국의 전통 음식입니다. 아래는 비빔밥을 만드는 3단계 레시피입니다.

### 1단계: 재료 준비하기
필요한 재료를 준비합니다.
- 밥: 1공기 (약 200g)
- 나물: 시금치, 고사리, 숙주나물 등 각각 50g씩
- 채소: 당근, 오이, 애호박 (각각 1/4개)
- 고기: 소고기(다진 것) 100g (옵션)
- 계란: 1개
- 고추장: 1큰술
- 참기름: 1큰술
- 소금, 후추: 적당량
- 깨: 약간

### 2단계: 재료 조리하기
1. **나물 조리**: 시금치, 고사리, 숙주나물은 각각 끓는 물에 살짝 데친 후 찬물에 헹구고 물기를 빼고, 소금과 참기름으로 간을 합니다.
2. **채소 준비**: 당근, 오이, 애호박은 얇게 채 썰고 팬에 기름을 두른 후 각각 볶아줍니다. 소금으로 간을 맞춥니다.
3. **고기 조리**: 다진 소고기는 팬에 볶아 소금과 후추로 간을 해줍니다.
4. **계란 조리**: 달궈진 팬에 기름을 약간 두르고 계란을 프라이하여 반숙으로 익힙니다.

### 3단계: 비빔밥 완성하기
1. 그릇에 밥을 담고, 준비한 나물, 채소, 고기, 계란을 고르게 올립니다.
2. 고추장과 참기름을 뿌리고, 깨를 뿌립니다.
3. 모든 재료를 잘 섞어 먹습니다. 취향에 따라 추가로 고추장이나 참기름을 더해도 좋습니다.

맛있게 드세요!


In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {subject} 전문 강사입니다."),
    ("human", "{topic}을 {level} 수준으로 설명해주세요.")
])

result = prompt.invoke({
    "subject": "인공지능",
    "topic": "LangChain",
    "level": "초보자"
})

for msg in result.to_messages():
    print(type(msg).__name__, ":", msg.content)

SystemMessage : 당신은 인공지능 전문 강사입니다.
HumanMessage : LangChain을 초보자 수준으로 설명해주세요.


**📝 정리** — `ChatPromptTemplate.invoke()`의 결과는 메시지 리스트(`ChatPromptValue`)다. 모델을 부르기 전에 이렇게 출력해 보면 프롬프트가 어떻게 채워졌는지 확인할 수 있다.

- PromptTemplate: 하나의 문장을 변수로 완성하는 템플릿
- ChatPromptTemplate: System, Human, AI 등의 역할을 구분하여 대화를 구성하는 템플릿

---
### 5. LCEL: 프롬프트 → 모델 → 문자열 파서

- 기존 `prompt | llm`은 `AIMessage`를 반환합니다. 뒤에 `StrOutputParser()`를 이어 붙이면 **문자열**을 바로 얻습니다. 
- 프롬프트와 출력 파서를 모델과 독립적으로 조합할 수 있다는 점이 LCEL의 장점입니다.

In [21]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

recipe_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 한국어 요리 강사입니다. 간결하게 설명하세요."),
    ("human", "{food} 조리법을 3단계로 알려주세요."),
])
recipe_chain = recipe_prompt | llm_openai | StrOutputParser()
answer = recipe_chain.invoke({"food": "비빔밥"})
print(type(answer))  # str
print(answer)


<class 'langchain_core.messages.base.TextAccessor'>
비빔밥 조리법을 3단계로 간단히 설명드리겠습니다.

1. **재료 준비**: 시금치, 당근, 애호박, 버섯, 고기(소고기 또는 닭고기) 등을 채 썰고, 각 재료는 따로 볶거나 데쳐서 간을 합니다. 밥도 준비합니다.

2. **비빔양념 만들기**: 고추장, 참기름, 설탕, 다진 마늘을 섞어 비빔양념을 만듭니다.

3. **비빔밥 완성**: 준비한 밥 위에 볶은 재료들을 예쁘게 올리고, 비빔양념을 곁들인 후, 계란 프라이를 올립니다. 잘 비벼서 맛있게 즐기세요!


**📝 정리** — `type(answer)`가 `str`이 아니라 `TextAccessor`로 나왔다. 확인해 보니 `str`을 상속한 클래스라서 문자열처럼 그대로 쓰면 된다.

### 6. 모델 교체만으로 동일 체인 재사용

체인의 가운데 모델만 교체하고 동일한 프롬프트 및 파서를 재사용합니다.


In [22]:
# Ollama가 준비된 경우에만 실행하세요.
local_recipe_chain = recipe_prompt | llm_exaone | StrOutputParser()
# print(local_recipe_chain.invoke({"food": "김치볶음밥"}))
print("Ollama 체인 구성 완료. 서버 및 모델을 준비한 뒤 주석을 해제하세요.")


Ollama 체인 구성 완료. 서버 및 모델을 준비한 뒤 주석을 해제하세요.


**📝 정리** — EXAONE 체인은 만들기만 하고 호출은 주석 처리돼 있다. 프롬프트와 파서는 그대로 두고 가운데 모델만 바꿨다.

### 핵심 API 한눈에 보기

| API | import | 역할 | 반환 유형 |
|---|---|---|---|
| `ChatOpenAI` | `langchain_openai` | OpenAI 채팅 모델 | `invoke()` → `AIMessage` |
| `ChatOllama` | `langchain_ollama` | 로컬 Ollama 모델 | `invoke()` → `AIMessage` |
| `SystemMessage`, `HumanMessage` | `langchain_core.messages` | 역할별 대화 입력 | 메시지 객체 |
| `PromptTemplate` | `langchain_core.prompts` | 문자열 프롬프트 템플릿 | `StringPromptValue` |
| `ChatPromptTemplate` | `langchain_core.prompts` | 메시지 프롬프트 템플릿 | `ChatPromptValue` |
| `StrOutputParser` | `langchain_core.output_parsers` | AI 응답에서 문자열 추출 | `str` |
| `prompt \| llm \| parser` | LCEL | Runnable들을 연결 | 마지막 구성요소에 따라 달라짐 |

공식 문서: https://docs.langchain.com/oss/python/langchain/overview


LangChain 공식 가이드에서는 \
단순한 체인에는 LCEL을, 복잡한 상태 관리와 반복 실행이 필요한 워크플로에는 LangGraph를 사용을 권장합니다.

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

#### [ 정리 ]

1. **LangChain ≠ 언어 모델**: 모델을 호출하고 연결하는 프레임워크입니다.
2. **`ChatOpenAI`는 채팅 모델 인터페이스**입니다. `gpt-4o-mini`에 구형 Completion용 `OpenAI` 클래스를 사용하지 않습니다.
3. **프롬프트 템플릿과 모델은 별개**이므로 템플릿을 재사용할 수 있습니다.
4. **LCEL `|`는 출력 → 입력 연결**입니다. 연결 단계마다 자료형을 확인해야 합니다.
5. **실제 답변 품질·실행 결과는 달라질 수 있습니다.** 같은 프롬프트에도 모델·temperature·버전 등에 따라 출력이 달라질 수 있습니다.

#### 참고 자료

- [LangChain Python 소개](https://docs.langchain.com/oss/python/langchain/overview)
- [PromptTemplate API](https://reference.langchain.com/python/langchain-core/prompts/prompt/PromptTemplate)
- [ChatPromptTemplate API](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate)
- [StrOutputParser API](https://reference.langchain.com/python/langchain-core/output_parsers/string/StrOutputParser)

## 🔁 복습 질문

<details>
<summary>1. <code>llm.invoke("질문")</code>의 결과 타입과 답 텍스트 위치는?</summary>

`AIMessage`, 답 텍스트는 `.content`.
</details>

<details>
<summary>2. <code>gpt-4o-mini</code>를 부를 때 <code>OpenAI</code> 클래스가 아니라 <code>ChatOpenAI</code>를 쓰는 이유는?</summary>

`OpenAI`는 문장 이어 쓰기용 옛 Completion 인터페이스이고, gpt-4o-mini는 역할이 있는 메시지를 받는 채팅 모델이기 때문이다.
</details>

<details>
<summary>3. <code>PromptTemplate</code>과 <code>ChatPromptTemplate</code>의 <code>invoke()</code> 결과는?</summary>

각각 `StringPromptValue`(문자열 프롬프트)와 `ChatPromptValue`(메시지 리스트).
</details>

<details>
<summary>4. 체인 끝에 <code>StrOutputParser()</code>를 붙이면 무엇이 달라지나?</summary>

`AIMessage` 대신 바로 문자열(`str`을 상속한 `TextAccessor`)을 돌려준다.
</details>
